## Transferring data between tables

This notebook allows the user to move the entries of specified columns from a source table into a target table. This is useful if a pipeline step had to be re-done due to a bug or methodological change, but you don't want to re-run the entire pipeline.

In [ ]:
from astropy.io import fits
import astropy.table as aptb
import numpy as np

from pathlib import Path

In [ ]:
SPEC_TYPE = "weight_skysub"

# specify the source and target file paths
tables_dir = Path("../megatables")
source_name = f"fit_catalog_qc_cpts_stack_zeldamcmc_refit_{SPEC_TYPE}.fits"
source_path = tables_dir / source_name
if not source_path.exists():
    raise FileNotFoundError(f"Source file not found: {source_path}")
target_name = f"fit_catalog_qc_cpts_stack_{SPEC_TYPE}.fits"
target_path = tables_dir / target_name
if not target_path.exists():
    raise FileNotFoundError(f"Target file not found: {target_path}")

# New name for the target (if needed)
new_target_name = f"fit_catalog_qc_cpts_stack_zeldamcmc_{SPEC_TYPE}.fits"
new_target_path = tables_dir / new_target_name

In [ ]:
# Specify columns to be updated in the target table
source_hdul = fits.open(source_path)
source_table = aptb.Table(source_hdul[1].data)
columns_to_update = [x for x in source_table.colnames if 'ZELDA' in x]

print(f"Columns to update: {columns_to_update}")

In [ ]:
# Open the tables
with fits.open(source_path) as source_hdul, fits.open(target_path) as target_hdul:
    source_table = aptb.Table(source_hdul[1].data)
    target_table = aptb.Table(target_hdul[1].data)

    # If the target table doesn't have the columns to update, add them with default values
    for col in columns_to_update:
        if col not in target_table.colnames:
            target_table[col] = np.full(len(target_table), np.nan)  # or any default value

    # Create a mapping from source table rows to target table rows based on both 'iden' and 'CLUSTER' columns
    source_iden_cluster = list(zip(source_table["iden"], source_table["CLUSTER"]))
    target_iden_cluster = list(zip(target_table["iden"], target_table["CLUSTER"]))
    mapping = {source_iden_cluster[i]: i for i in range(len(source_iden_cluster))}

    # Update the target table with values from the source table for the specified columns
    for i, target_row in enumerate(target_table):
        key = (target_row["iden"], target_row["CLUSTER"])
        if key in mapping:
            source_row = source_table[mapping[key]]
            for col in columns_to_update:
                target_table[col][i] = source_row[col]

    # Save the updated target table back to a new FITS file
    target_table.write(new_target_path, overwrite=True)